In [1]:
import brightway2 as bw
from brightway2 import *
from premise import *
import os
import wurst
import time
import copy

In [2]:
startTime = time.time() # just to see how much time the code takes to run (this is the start)

In [3]:
bw.projects.set_current('Prospective LCA v2') # set current project

In [4]:
bioSphereDBName = 'biosphere3'
bioSphereDB = bw.Database(bioSphereDBName) # import the biosphere database

In [5]:
allDBNames = list(bw.databases)

In [6]:
myDBNames = []
for DBName in allDBNames:
    if 'Abhi' in DBName:
        myDBNames.append(DBName)

if len(myDBNames) > 0:
    for i in range(0, len(myDBNames)):
        del bw.databases[myDBNames[i]]

allDBNames = list(bw.databases)

In [7]:
ecoinventSSP2DBNames = []
for DBName in allDBNames:
    if 'SSP2' in DBName:
        ecoinventSSP2DBNames.append(DBName)
ecoinventSSP2DBNames.sort()

In [12]:
for ecoinventSSP2DBName in ecoinventSSP2DBNames:
    
    ecoinventSSP2DB = bw.Database(ecoinventSSP2DBName)
    mySSP2DBName = ecoinventSSP2DBName.replace('ecoinvent 3.8 cutoff', 'lci-Abhi')
    mySSP2DB = bw.Database  (mySSP2DBName)
    mySSP2DB.register()

    solarElecAct = [activity for activity in ecoinventSSP2DB
                    if 'electricity production, photovoltaic, 570kWp open ground installation, multi-Si' in activity['name']
                    and 'electricity, low voltage' in activity['reference product']
                    and 'RoW' in activity['location']][0]
    

    hydrogenBAUAct = [activity for activity in ecoinventSSP2DB 
                      if 'hydrogen production, steam methane reforming of natural gas, 25 bar' in activity['name']
                      and 'World' in activity['location']][0]
    hydrogenBAUActCopy = hydrogenBAUAct.copy(database = mySSP2DB.name)

    hydrogenBlueAct = [activity for activity in ecoinventSSP2DB 
                       if r'hydrogen production, steam methane reforming of natural gas, with CCS (MDEA, 98% eff.), 25 bar' in activity['name']
                       and 'World' in activity['location']][0]
    hydrogenBlueActCopy = hydrogenBlueAct.copy(database = mySSP2DB.name)

    hydrogenPEMGridAct = [activity for activity in ecoinventSSP2DB 
                          if 'hydrogen production, gaseous, 200 bar, from PEM electrolysis, from grid electricity' in activity['name']
                          and 'World' in activity['location']][0]
    hydrogenPEMGridActCopy = hydrogenPEMGridAct.copy(database = mySSP2DB.name)

    hydrogenPEMSolarCopy = hydrogenPEMGridActCopy.copy(
        name = 'hydrogen production, gaseous, 200 bar, from PEM electrolysis, from solar electricity')
    
    elecHydrogenPEMSolarCopy = [exchange for exchange in hydrogenPEMSolarCopy.technosphere()
                                if 'electricity, low voltage' in exchange['product']][0]
    
    elecHydrogenPEMSolarCopy.input = solarElecAct
    elecHydrogenPEMSolarCopy.save()